# Results Notebook Demo

This notebook mirrors `scripts/notebook_results_demo.py` and recreates the legacy upload flow in a notebook-friendly format.

It works in dry-run mode when `OWI_METADATABASE_API_TOKEN` is not set.

In [3]:
from __future__ import annotations

import enum
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

if not hasattr(enum, "StrEnum"):
    class _CompatStrEnum(str, enum.Enum):
        pass
    enum.StrEnum = _CompatStrEnum

workspace_root = Path.cwd().resolve().parent
sys.path.insert(0, str(workspace_root / "src"))
sys.path.insert(0, str(workspace_root.parent / "owi-metadatabase-sdk" / "src"))

from owi.metadatabase.geometry.io import GeometryAPI
from owi.metadatabase.locations.io import LocationsAPI
from owi.metadatabase.results import (
    AnalysisDefinition,
    RelatedObject,
    ResultSeries,
    ResultVector,
    ResultsAPI,
)
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer

DEFAULT_BASE_URL = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"


In [4]:
BASE_URL = DEFAULT_BASE_URL
SITE = "Belwind"
LOCATION = "BBC01"
ANALYSIS_NAME = "Example analysis"
TOKEN = os.getenv(TOKEN_ENV_VAR)
UPLOAD_ENABLED = TOKEN is not None

print({"base_url": BASE_URL, "site": SITE, "location": LOCATION, "upload_enabled": UPLOAD_ENABLED})


{'base_url': 'https://owimetadatabase-dev.azurewebsites.net/api/v1', 'site': 'Belwind', 'location': 'BBC01', 'upload_enabled': False}


In [5]:
def resolve_context_ids(api_root: str, token: str, site: str, location: str) -> tuple[int | None, int | None, int | None]:
    locations_api = LocationsAPI(api_root=api_root, token=token)
    geometry_api = GeometryAPI(api_root=api_root, token=token)
    site_id = locations_api.get_projectsite_detail(projectsite=site)["id"]
    location_id = locations_api.get_assetlocation_detail(projectsite=site, assetlocation=location)["id"]
    subassemblies = geometry_api.get_subassemblies(projectsite=site, assetlocation=location)["data"]
    subassembly_id = int(subassemblies.loc[0, "id"]) if not subassemblies.empty else None
    return site_id, location_id, subassembly_id


site_id = 1
location_id = None
related_object = RelatedObject(type="geometry.subassembly", id=1)

if UPLOAD_ENABLED:
    site_id, location_id, subassembly_id = resolve_context_ids(BASE_URL, TOKEN, SITE, LOCATION)
    related_object = RelatedObject(type="geometry.subassembly", id=subassembly_id) if subassembly_id else None

display(pd.DataFrame([{"site_id": site_id, "location_id": location_id, "related_object": related_object.model_dump() if related_object else None}]))


,site_id,location_id,related_object
0,1,None,"{'type': 'geometry.subassembly', 'id': 1}"


In [6]:
analysis = AnalysisDefinition(
    name=ANALYSIS_NAME,
    source_type="notebook",
    description="Notebook reimplementation of the legacy notebook flow.",
    additional_data={"script": "notebook_results_demo.ipynb", "created_at": datetime.now(timezone.utc).isoformat()},
)

x_values = [float(index) for index in range(25)]
y_values = [float(index) for index in range(25)]

result_series = ResultSeries(
    analysis_name=ANALYSIS_NAME,
    analysis_kind="comparison",
    result_scope="site",
    short_description="test_example_1",
    description="Example data uploaded from the notebook demo.",
    site_id=site_id,
    location_id=location_id,
    related_object=related_object,
    data_additional={"site_name": SITE, "location_name": LOCATION},
    vectors=[
        ResultVector(name="x", unit="mm", values=x_values),
        ResultVector(name="y", unit="mm", values=y_values),
    ],
)

analysis_payload = DjangoAnalysisSerializer().to_payload(analysis)
result_payload_preview = result_series.to_record_payload(analysis_id=0)

display(pd.DataFrame([analysis_payload]))
display(pd.DataFrame([result_payload_preview]))


,name,source_type,description,additional_data
0,Example analysis,notebook,Notebook reimplementation of the legacy notebo...,"{'script': 'notebook_results_demo.ipynb', 'cre..."


,analysis,site,location,short_description,description,data_additional,related_object,name_col1,units_col1,value_col1,name_col2,units_col2,value_col2,name_col3,units_col3,value_col3
0,0,1,None,test_example_1,Example data uploaded from the notebook demo.,"{'analysis_name': 'Example analysis', 'analysi...","{'type': 'geometry.subassembly', 'id': 1}",x,mm,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...",y,mm,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...",None,None,None


## Optional Live Upload

Run the next cell only when the token is configured and you want to create records on the configured server.

In [7]:
if not UPLOAD_ENABLED:
    print("Dry-run mode: set OWI_METADATABASE_API_TOKEN to enable uploads.")
else:
    api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
    created_analysis = api.create_analysis(analysis_payload)
    analysis_id = int(created_analysis["id"])
    result_payload = DjangoResultSerializer().to_payload(result_series, analysis_id=analysis_id)
    created_result = api.create_result(result_payload)

    try:
        retrieved_results = api.get_results(
            short_description=result_series.short_description,
            analysis=ANALYSIS_NAME,
        )["data"]
    except Exception:
        retrieved_results = created_result["data"]

    display(created_analysis["data"])
    display(retrieved_results.head())


Dry-run mode: set OWI_METADATABASE_API_TOKEN to enable uploads.
